# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 03 · Modelos clásicos

Entrena o audita el contrato de etiquetas v2.1 sin consultar test para seleccionar modelos, épocas o umbrales.

La suite representa texto mediante TF–IDF [1] y compara regresión logística [2], SVM lineal [3], Complement Naive Bayes [4] y descenso de gradiente estocástico [5]. La implementación usa scikit-learn [6] bajo una transformación uno-contra-resto para el problema multietiqueta [7]; la elección de n-gramas, pesos de clase e hiperparámetros es local.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import notebook_progress, run_with_progress, show_callout, show_command, show_result, show_summary, show_table
OPERATIONAL_PROMPT=ROOT/'config/prompt_operacional_ollama_v3_2.md'
if not OPERATIONAL_PROMPT.is_file():
    raise FileNotFoundError(f'Falta el prompt operacional vigente: {OPERATIONAL_PROMPT}')
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local', 'prompt_operacional': OPERATIONAL_PROMPT}, tone='success')


## Restauración reproducible del dataset

In [ ]:
from moderacion_peru.colab import prepare_local_bundle_input

if globals().get('COLAB_CONTEXT') is None:
    dataset_checkpoint = prepare_local_bundle_input('dataset_5_salidas', project_root=ROOT)
else:
    dataset_path = COLAB_CONTEXT.input('dataset_5_salidas')
    dataset_checkpoint = {
        'status': 'verified_in_colab',
        'input_key': 'dataset_5_salidas',
        'path': dataset_path,
        'bytes': dataset_path.stat().st_size,
    }
show_result('Dataset descomprimido y verificado', dataset_checkpoint, tone='success')


## Configuración y ejecución

In [ ]:
from moderacion_peru.experiments import train_classical_experiments
DATA=ROOT/'datos/model_ready/v2/dataset_5_salidas.jsonl'
OUTPUT=ROOT/'modelos/v2/clasicos'
SAFE_TO_DAMAGE_RATIO=4.0  # política fija en train y validation
PARALLEL_WORKERS=4  # 4/16 hilos: comparte la matriz dispersa sin saturar RAM
RUN_TRAINING=False
RUN_CHANNEL_ROBUSTNESS=False
if RUN_TRAINING:
    classical_result=run_with_progress('Suite clásica',train_classical_experiments,DATA,OUTPUT,variants=('base','policy_informed'),safe_to_damage_ratio=SAFE_TO_DAMAGE_RATIO,parallel_workers=PARALLEL_WORKERS,progress_unit='etapa')
    show_result('Clásicos base e informados por política',classical_result,tone='success')
if RUN_CHANNEL_ROBUSTNESS:
    robustness_result=run_with_progress('Robustez por canal',train_classical_experiments,DATA,OUTPUT/'channel_heldout',model_names=('logistic_regression',),variants=('base','policy_informed'),safe_to_damage_ratio=SAFE_TO_DAMAGE_RATIO,split_scheme='channel',parallel_workers=PARALLEL_WORKERS,progress_unit='etapa')
    show_result('Robustez con canales retenidos',robustness_result,tone='success')
if not (RUN_TRAINING or RUN_CHANNEL_ROBUSTNESS):
    show_summary('Entrenamiento desactivado',{'salidas':'22 enmascaradas','SEGURO_train_validation':'4:1','TF-IDF':'una extracción por variante, reutilizada por cinco modelos','paralelismo':f'{PARALLEL_WORKERS} hilos compartiendo matriz dispersa','progreso':'barra por preparación, TF-IDF, candidato y validation','test':'natural completo, sellado'},tone='neutral')

## Referencias

[1] G. Salton and C. Buckley, "Term-Weighting Approaches in Automatic Text Retrieval," Inf. Process. Manage., vol. 24, no. 5, pp. 513–523, 1988, doi: 10.1016/0306-4573(88)90021-0.

[2] D. R. Cox, "The Regression Analysis of Binary Sequences," J. Roy. Stat. Soc. B, vol. 20, no. 2, pp. 215–232, 1958, doi: 10.1111/j.2517-6161.1958.tb00292.x.

[3] C. Cortes and V. Vapnik, "Support-Vector Networks," Mach. Learn., vol. 20, pp. 273–297, 1995, doi: 10.1007/BF00994018.

[4] J. D. M. Rennie, L. Shih, J. Teevan, et al., "Tackling the Poor Assumptions of Naive Bayes Text Classifiers," in Proc. ICML, 2003, pp. 616–623. [Online]. Available: https://people.csail.mit.edu/jrennie/papers/icml03-nb.pdf

[5] L. Bottou, "Large-Scale Machine Learning with Stochastic Gradient Descent," in Proc. COMPSTAT, 2010, pp. 177–186, doi: 10.1007/978-3-7908-2604-3_16.

[6] F. Pedregosa, G. Varoquaux, A. Gramfort, et al., "Scikit-Learn: Machine Learning in Python," J. Mach. Learn. Res., vol. 12, pp. 2825–2830, 2011. [Online]. Available: https://www.jmlr.org/papers/v12/pedregosa11a.html

[7] G. Tsoumakas and I. Katakis, "Multi-Label Classification: An Overview," Int. J. Data Warehousing Mining, vol. 3, no. 3, pp. 1–13, 2007, doi: 10.4018/jdwm.2007070101.